# XLAV — E3: prompt engineering cho text prompt (CLIPSeg, PhraseCut miniv)

Câu hỏi: **cách viết câu lệnh** có làm thay đổi chất lượng mask không? Mỗi query được chạy với
4 template cố định từ trước: `{phrase}`, `a photo of {phrase}`, `the {phrase}`, `{phrase} in the image`.
Ảnh, GT, ngưỡng 0,3 và metric giữ nguyên như E1; chỉ chuỗi text thay đổi.

1. Bật **GPU T4 x2** và **Internet** trong Settings của notebook Kaggle.
2. **Run All**. Kết quả ở `/kaggle/working/clipseg_e3_prompt_templates`, kèm file ZIP.

Báo cáo **mọi template**, cả tốt lẫn kém. `{phrase}` là gốc để so sánh từng cặp (ΔIoU, CI 95% bootstrap theo ảnh).
Không chọn template "tốt nhất" trên evaluation rồi coi là kết quả độc lập; ngưỡng bổ sung chỉ chọn trên calibration.
Chọn 100 query bằng seed 42 và chia calibration/evaluation theo image_id giống hệt E1.


## 1. Dependency

Giữ PyTorch/CUDA, NumPy, Pillow, scikit-learn và matplotlib sẵn có của Kaggle.
Cell này chỉ pin Transformers và gdown; không cài wheel PyTorch CPU.
Nếu kernel đã import Transformers phiên bản khác trước đó, restart session rồi chạy lại từ đầu.


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==4.55.4", "gdown==5.2.0",
])


## 2. Cấu hình

`MAX_QUERIES=100` theo kế hoạch; đặt `8` để kiểm tra nhanh. `BATCH_SIZE` là số prompt mỗi lần forward
(query × template trên cùng ảnh); giảm nếu thiếu VRAM. `DATA_ROOT=None` tự tìm trong `/kaggle/input`
rồi tải từ nguồn chính thức nếu chưa có.

`DECODER_CHECKPOINT=None` dùng decoder pretrained. Để đánh giá decoder đã train theo `EPOCHS` + `BATCH_SIZE`:
chạy notebook E1 với `TRAIN_DECODER=True`, lưu version để Output thành dataset, Add Input dataset đó vào notebook này,
rồi trỏ `DECODER_CHECKPOINT` tới `checkpoints/decoder_epoch_XXX.pt` của **epoch cuối**.
Kết quả ghi đè thư mục output; tải ZIP của lượt pretrained trước khi chạy lượt có checkpoint.


In [ ]:
from pathlib import Path

MAX_QUERIES = 100                # kế hoạch E3: 100 query; 8 = smoke
TEMPLATES = ["{phrase}", "a photo of {phrase}", "the {phrase}", "{phrase} in the image"]
BASELINE_TEMPLATE = "{phrase}"
BATCH_SIZE = 16                  # số prompt mỗi forward
OUTPUT_NAME = "clipseg_e3_prompt_templates"

# None = decoder pretrained. Hoặc decoder đã train trong notebook E1 theo EPOCHS + BATCH_SIZE, ví dụ
# "/kaggle/input/<output-e1>/clipseg_e1/checkpoints/decoder_epoch_004.pt"
DECODER_CHECKPOINT = None
DATA_ROOT = None                 # ví dụ: "/kaggle/input/phrasecut-miniv/VGPhraseCut_v0"
EVAL_SPLIT = "miniv"             # "miniv" | "val" | "test"; support của E2 lấy trong cùng split
MODEL_PATH = None                # thư mục checkpoint offline, hoặc None
IMAGE_ROOT = None                # thư mục chứa ảnh <image_id>.jpg; None = tìm trong mọi Kaggle Input
WORK_ROOT = Path("/kaggle/working")
REQUIRE_GPU = True               # False chỉ khi kiểm tra trên máy CPU
USE_AMP = False                  # mặc định FP32 như E1
SEED = 42
THRESHOLD = 0.3
CALIBRATION_FRACTION = 0.2
THRESHOLD_GRID = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
N_BOOTSTRAP = 2000               # bootstrap theo ảnh cho CI 95% của ΔIoU
TIE_TOLERANCE = 0.01             # |ΔIoU| ≤ 0,01 tính là hòa
SAVE_MASKS = True
N_VISUALIZATIONS = 6
DOWNLOAD_WORKERS = 4

MODEL_ID = "CIDAS/clipseg-rd64-refined"
MODEL_REVISION = "999e0328d9e10b484360c477313983f9afdd7050"
PHRASECUT_API_COMMIT = "276470b7773d8a230aee5b5536876fbc485fb04b"


In [ ]:
import csv
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import shutil
import time
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext

import gdown
import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
from PIL import Image, ImageDraw
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm
from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor
from IPython.display import display

assert 0 < CALIBRATION_FRACTION < 1
assert 0 <= THRESHOLD <= 1 and all(0 <= t <= 1 for t in THRESHOLD_GRID)
assert N_BOOTSTRAP >= 100 and TIE_TOLERANCE >= 0
random.seed(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Chưa có CUDA. Bật GPU trong Kaggle Settings rồi restart session.")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    try:
        torch.ones(1, device=device).add_(1).item()
    except RuntimeError as exc:
        raise RuntimeError("PyTorch của Kaggle không chạy được trên GPU này (thường gặp với P100). "
                           "Chọn Accelerator GPU T4 x2 rồi restart session.") from exc
assert not USE_AMP or device.type == "cuda", "FP16 AMP yêu cầu CUDA."
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUT = WORK_ROOT / OUTPUT_NAME
OUT.mkdir(parents=True, exist_ok=True)
# Xóa kết quả lượt cũ để không trộn artifact vào ZIP.
for folder in ["masks", "visualizations"]:
    shutil.rmtree(OUT / folder, ignore_errors=True)
for filename in ["summary.json", "run.json"]:
    (OUT / filename).unlink(missing_ok=True)
(WORK_ROOT / f"{OUTPUT_NAME}.zip").unlink(missing_ok=True)

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")

def write_csv(path, rows):
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

versions = {name: importlib.metadata.version(name) for name in
            ["torch", "transformers", "numpy", "Pillow", "scikit-learn", "matplotlib", "gdown"]}
print("Device:", device, "| GPU:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
print(json.dumps(versions, indent=2))


## 3. Lấy annotation chính thức và chọn query

Chỉ tải `refer_<EVAL_SPLIT>.json` và metadata ảnh từ Drive ID trong
[downloader tác giả](https://github.com/ChenyunWu/PhraseCutDataset/blob/276470b7773d8a230aee5b5536876fbc485fb04b/download_dataset.py).
Chỉ ảnh của các query được chọn mới được tải, không tải toàn bộ Visual Genome. Với `test`/`val`
(khoảng 2.500–2.900 ảnh) nên Add Input dataset đã có ảnh để khỏi tải lại mỗi lượt.
Nếu Google Drive hết quota, Add Input dữ liệu đã tải rồi đặt `DATA_ROOT`.

Chia calibration/evaluation theo **image_id**, seed 42, trước khi đọc prediction.
Nếu giới hạn số query, sampling bằng seed, lưu đúng danh sách vào `selected_tasks.json`.


In [ ]:
# Drive ID của annotation chính thức, theo download_dataset.py của PhraseCut.
REFER_DRIVE_IDS = {"miniv": "1oLcTQ1blTIQyu5ZMelQN2HSniuQaxM4E", "val": "1UyojArOFPlsSeNbA9fHWjCjOOU-OCohG",
                   "test": "1jrzXm1gcq6f5hNDeamZd0UmyyHUv61IZ", "train": "1qx-0q6r9r0YUGpoyT0B8HJKmUFWQDSu7"}

def metadata_path(root):
    for name in ["image_data_split.json", "image_data_split3000.json"]:
        candidate = root / name
        if candidate.is_file():
            return candidate
    return root / "image_data_split.json"

if DATA_ROOT is None:
    candidates = sorted(Path("/kaggle/input").rglob(f"refer_{EVAL_SPLIT}.json")) if Path("/kaggle/input").exists() else []
    candidates = [p.parent for p in candidates if metadata_path(p.parent).is_file()]
    if len(candidates) > 1:
        raise ValueError(f"Có nhiều dataset chứa refer_{EVAL_SPLIT}.json; đặt DATA_ROOT rõ ràng: " + str(candidates))
    data_root = candidates[0] if candidates else WORK_ROOT / "phrasecut_annotations"
else:
    data_root = Path(DATA_ROOT)
    if not data_root.is_dir():
        raise FileNotFoundError(f"DATA_ROOT không tồn tại: {data_root}")
data_root.mkdir(parents=True, exist_ok=True)
annotation_path = data_root / f"refer_{EVAL_SPLIT}.json"
info_path = metadata_path(data_root)

def ensure_json(path, file_id):
    if path.is_file():
        value = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError(f"JSON không phải danh sách annotation: {path}")
        return value
    if str(path).startswith("/kaggle/input/"):
        raise FileNotFoundError(f"Thiếu {path.name} trong input read-only; bổ sung Kaggle Dataset.")
    temp = path.with_suffix(".download")
    try:
        result = gdown.download(id=file_id, output=str(temp), quiet=False)
        if result is None:
            raise RuntimeError("Google Drive không trả về file.")
        value = json.loads(temp.read_text(encoding="utf-8"))
        if not isinstance(value, list) or not value:
            raise ValueError("Nội dung download không phải annotation JSON.")
        temp.replace(path)
        return value
    except Exception as exc:
        raise RuntimeError(f"Không tải được {path.name}. Bật Internet, hoặc Add Input dataset và đặt DATA_ROOT.") from exc

tasks = ensure_json(annotation_path, REFER_DRIVE_IDS[EVAL_SPLIT])
image_info = ensure_json(info_path, "1xB9eExJo35K3DQu8PnBWZ1q_OMoMl8-i")
info_by_id = {int(row["image_id"]): row for row in image_info if row["split"] == EVAL_SPLIT}
assert len({str(t["task_id"]) for t in tasks}) == len(tasks), "Task ID bị trùng."
assert all(int(t["image_id"]) in info_by_id for t in tasks), f"Annotation không thuộc split {EVAL_SPLIT}."
assert all(t.get("Polygons") and t.get("phrase", "").strip() for t in tasks), "Thiếu polygon GT hoặc phrase."
assert all(re.fullmatch(r"[A-Za-z0-9_-]+", str(t["task_id"])) for t in tasks), "Task ID không hợp lệ."

selected = sorted(tasks, key=lambda t: str(t["task_id"]))
if MAX_QUERIES is not None and MAX_QUERIES < len(selected):
    selected = random.Random(SEED).sample(selected, MAX_QUERIES)
selected.sort(key=lambda t: (int(t["image_id"]), str(t["task_id"])))
image_ids = sorted({int(t["image_id"]) for t in selected})
if len(image_ids) < 2:
    raise ValueError("Cần ít nhất 2 image_id để chia calibration/evaluation; tăng MAX_QUERIES.")
shuffled_ids = image_ids.copy()
random.Random(SEED).shuffle(shuffled_ids)
n_cal = min(len(image_ids) - 1, max(1, round(len(image_ids) * CALIBRATION_FRACTION)))
calibration_ids = set(shuffled_ids[:n_cal])
group_for_image = {i: "calibration" if i in calibration_ids else "evaluation" for i in image_ids}
assert not calibration_ids.intersection(set(image_ids) - calibration_ids)
write_json(OUT / "selected_tasks.json", [{"task_id": str(t["task_id"]), "image_id": int(t["image_id"]),
    "phrase": t["phrase"], "group": group_for_image[int(t["image_id"])]} for t in selected])
print(f"Annotation {EVAL_SPLIT}: {len(tasks)} queries / {len(info_by_id)} images")
print(f"Selected: {len(selected)} queries / {len(image_ids)} images; calibration {n_cal} images")


## 4. Tải và kiểm tra ảnh

Ưu tiên ảnh đã có trong dataset input. Ảnh thiếu sẽ tải từ URL Visual Genome trong metadata
bằng HTTPS vào working cache, retry có giới hạn. Lỗi tải hoặc kích thước khác metadata làm
dừng lượt chạy: không âm thầm bỏ query để tạo số liệu đẹp hơn.


In [ ]:
image_cache = WORK_ROOT / "phrasecut_image_cache"
image_cache.mkdir(parents=True, exist_ok=True)
def image_search_roots(*extra):
    """Nơi tìm ảnh <image_id>.jpg: IMAGE_ROOT nếu đặt, nếu không thì mọi Kaggle Input; cộng thư mục annotation."""
    roots = [Path(IMAGE_ROOT)] if IMAGE_ROOT else ([Path("/kaggle/input")] if Path("/kaggle/input").exists() else [])
    roots += [Path(r) for r in extra]
    # Bỏ thư mục nằm trong thư mục đã quét để không quét hai lần.
    return [r for i, r in enumerate(roots) if r.is_dir() and not any(r == q or q in r.parents for q in roots[:i])]

def index_local_images(roots, wanted_ids):
    index = {}
    for search_root in roots:
        for path in sorted(search_root.rglob("*")):
            if path.suffix.lower() in {".jpg", ".jpeg", ".png"} and path.stem.isdigit() and int(path.stem) in wanted_ids:
                index.setdefault(int(path.stem), path)
    return index

local_images = index_local_images(image_search_roots(data_root), set(image_ids))
print(f"Ảnh có sẵn trong Input: {len(local_images)}/{len(image_ids)}; phần còn lại tải từ Visual Genome")

def valid_image(path, info):
    with Image.open(path) as image:
        image.load()
        if image.size != (int(info["width"]), int(info["height"])):
            raise ValueError(f"{path.name}: ảnh {image.size} != metadata {(info['width'], info['height'])}")

def prepare_image(image_id):
    info = info_by_id[image_id]
    path = local_images.get(image_id, image_cache / f"{image_id}.jpg")
    if path.is_file():
        valid_image(path, info)
        return image_id, path
    url = info["url"].replace("http://", "https://", 1)
    temp = path.with_suffix(".download")
    for attempt in range(3):
        try:
            with requests.get(url, stream=True, timeout=(10, 60)) as response:
                response.raise_for_status()
                with temp.open("wb") as f:
                    for chunk in response.iter_content(1024 * 1024):
                        f.write(chunk)
            valid_image(temp, info)
            temp.replace(path)
            return image_id, path
        except Exception:
            if attempt == 2:
                raise
            time.sleep(attempt + 1)

image_paths, image_errors = {}, []
def prepare_safe(image_id):
    try:
        return prepare_image(image_id), None
    except Exception as exc:
        return None, {"image_id": image_id, "error": str(exc)}

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    for result, error in tqdm(pool.map(prepare_safe, image_ids), total=len(image_ids), desc="Images"):
        if error:
            image_errors.append(error)
        else:
            image_paths[result[0]] = result[1]
write_json(OUT / "image_errors.json", image_errors)
if image_errors:
    raise RuntimeError(f"{len(image_errors)} ảnh lỗi; xem image_errors.json. Sửa dữ liệu rồi chạy lại.")
print("Validated images:", len(image_paths))


## 5. Nạp model, inference và metric

Nếu `DECODER_CHECKPOINT` được đặt, decoder được thay bằng decoder đã train trong notebook E1
(CLIP vẫn là backbone đóng băng của checkpoint gốc). Epoch, batch size và lịch sử loss đọc từ checkpoint
được ghi vào `summary.json` và `run.json`. E1 chỉ train trên split `train`, nên miniv vẫn là dữ liệu chưa thấy.

Model, tiền xử lý ảnh (352×352 như CLIPSeg gốc), rasterize GT và metric giống hệt E1, gồm cả bộ metric
theo protocol paper (`paper_protocol`: miou_0.3, fgiou_0.3, AP ở 352×352). `predict` nhận **hoặc** danh sách text prompt
**hoặc** danh sách ảnh support (truyền vào `conditional_pixel_values` của CLIPSeg),
luôn dự đoán trên cùng một ảnh query. Logits nội suy bilinear về kích thước gốc rồi sigmoid.

`paired_comparison` so từng query với biến thể gốc: ΔIoU = biến thể − gốc, CI 95% bằng
bootstrap theo **ảnh** (các query cùng ảnh không độc lập), kèm số query thắng/hòa/thua.


In [ ]:
load_kwargs = {"local_files_only": True} if MODEL_PATH else {"revision": MODEL_REVISION}
model_source = str(MODEL_PATH) if MODEL_PATH else MODEL_ID
processor = CLIPSegProcessor.from_pretrained(model_source, use_fast=False, **load_kwargs)
model = CLIPSegForImageSegmentation.from_pretrained(model_source, **load_kwargs).to(device).eval()
assert processor.image_processor.size == {"height": 352, "width": 352}, "Processor không có input 352 như baseline."
assert model.config.vision_config.patch_size == 16, "Baseline yêu cầu ViT-B/16."
print("Model:", model_source, "| parameters:", sum(p.numel() for p in model.parameters()))

try:
    from skimage.draw import polygon2mask
except ImportError as exc:
    raise ImportError("Cần scikit-image (có sẵn trên Kaggle); cài bằng pip install scikit-image.") from exc
try:
    from scipy.integrate import simpson
except ImportError:  # SciPy cũ
    from scipy.integrate import simps as simpson

# Tiền xử lý và độ phân giải chấm điểm theo score.py + datasets/phrasecut.py của CLIPSeg.
PAPER_SIZE = 352
PAPER_THRESHOLDS = np.linspace(0, 1, 51)[1:-1]   # 49 ngưỡng 0,02–0,98 như FixedIntervalMetrics
image_mean = torch.tensor(processor.image_processor.image_mean, dtype=torch.float32).view(3, 1, 1)
image_std = torch.tensor(processor.image_processor.image_std, dtype=torch.float32).view(3, 1, 1)

def rasterize_gt(task, width, height, allow_empty=False):
    """Hợp polygon mọi instance bằng skimage polygon2mask (x, y đảo thành hàng, cột), như PhraseCut.load_sample."""
    target = np.zeros((height, width), dtype=bool)
    for instance in task["Polygons"]:
        for polygon in instance:
            if len(polygon) >= 3:
                target |= polygon2mask((height, width), np.asarray(polygon, dtype=np.float64)[:, ::-1])
    if not allow_empty and not target.any():
        raise ValueError(f"GT rỗng: task {task['task_id']}")
    return target

def paper_pixel_values(image):
    """Ảnh RGB → [3,352,352]: resize bilinear align_corners=True trên ảnh gốc, /255, chuẩn hóa mean/std."""
    pixels = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1).unsqueeze(0)
    pixels = F.interpolate(pixels, size=(PAPER_SIZE, PAPER_SIZE), mode="bilinear", align_corners=True)[0]
    return (pixels / 255 - image_mean) / image_std

def paper_target(target):
    """GT ở độ phân giải gốc → 352×352 bằng nearest, như nnf.interpolate trong load_sample."""
    mask = torch.from_numpy(target.astype(np.uint8))[None, None]
    return F.interpolate(mask, size=(PAPER_SIZE, PAPER_SIZE), mode="nearest")[0, 0].numpy().astype(bool)

def paper_threshold_stats(probability, target):
    """tp, fp, fn, tn tại 49 ngưỡng (dương khi p > t) cho một query, như FixedIntervalMetrics.add."""
    order = np.argsort(probability, axis=None, kind="stable")
    p, g = probability.ravel()[order], target.ravel()[order]
    below = np.searchsorted(p, PAPER_THRESHOLDS, side="right")   # số pixel p <= t
    fn = np.concatenate([[0], np.cumsum(g, dtype=np.int64)])[below]
    tp = int(g.sum()) - fn
    fp = (len(p) - below) - tp
    tn = below - fn
    return np.stack([tp, fp, fn, tn]).astype(np.int64)

def paper_metrics(stats):
    """Metric bảng PhraseCut của paper từ stats [Q,4,49]: miou (trung bình theo query), fgiou (cộng dồn), AP Simpson."""
    stats = np.asarray(stats, dtype=np.float64)
    tp, fp, fn = stats[:, 0], stats[:, 1], stats[:, 2]
    s_tp, s_fp, s_fn = tp.sum(0), fp.sum(0), fn.sum(0)
    made = (s_tp + s_fp) > 0
    pairs = sorted(set(zip((s_tp / (1 + s_tp + s_fn))[made], (s_tp / (1 + s_tp + s_fp))[made])))
    recalls, precisions = (np.array(v) for v in zip(*pairs))
    fgiou = s_tp / (1 + s_tp + s_fp + s_fn)
    miou = (tp / (1 + tp + fp + fn)).mean(0)
    index = {t: int(np.argmin(np.abs(PAPER_THRESHOLDS - t))) for t in (0.3, 0.5)}
    return {"n_queries": len(stats), "miou_0.3": float(miou[index[0.3]]), "fgiou_0.3": float(fgiou[index[0.3]]),
            "fgiou_0.5": float(fgiou[index[0.5]]), "ap": float(simpson(precisions, x=recalls)),
            "miou_best": float(miou.max()), "miou_best_t": float(PAPER_THRESHOLDS[miou.argmax()]),
            "fgiou_best": float(fgiou.max()), "fgiou_best_t": float(PAPER_THRESHOLDS[fgiou.argmax()])}

def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize(device)
LORA_GRAD_CHECKPOINT = False
import math
from contextlib import contextmanager
import torch.utils.checkpoint

class LoRALinear(torch.nn.Module):
    """W·x + (alpha/r)·B·A·x; B khởi tạo 0 nên lúc đầu đúng bằng lớp gốc. Lớp gốc giữ đóng băng."""

    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base = base
        self.lora_a = torch.nn.Parameter(torch.empty(rank, base.in_features, device=base.weight.device))
        self.lora_b = torch.nn.Parameter(torch.zeros(base.out_features, rank, device=base.weight.device))
        torch.nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))
        self.scale = alpha / rank
        self.enabled = True

    def forward(self, x):
        out = self.base(x)
        if self.enabled:
            out = out + x @ self.lora_a.t().to(x.dtype) @ self.lora_b.t().to(x.dtype) * self.scale
        return out

def lora_layers(model):
    """Các layer CLIP vision có output đi vào decoder (0..max(extract_layers))."""
    return list(model.clip.vision_model.encoder.layers[:max(model.extract_layers) + 1])

def lora_modules(model):
    return [m for m in model.clip.vision_model.modules() if isinstance(m, LoRALinear)]

def has_lora(model):
    return bool(lora_modules(model))

def attach_lora(model, rank, alpha):
    assert not has_lora(model), 'LoRA đã được gắn'
    for layer in lora_layers(model):
        for name in ['q_proj', 'v_proj']:
            setattr(layer.self_attn, name, LoRALinear(getattr(layer.self_attn, name), rank, alpha))

def lora_parameters(model):
    return [p for m in lora_modules(model) for p in (m.lora_a, m.lora_b)]

def trainable_groups(model, lr):
    """Nhóm 0 = decoder (lr), nhóm 1 = LoRA (lr × LORA_LR_SCALE). Không có LoRA thì chỉ một nhóm, như B1."""
    groups = [{'params': list(model.decoder.parameters()), 'lr': lr}]
    if lora_parameters(model):
        groups.append({'params': lora_parameters(model), 'lr': lr * LORA_LR_SCALE})
    return groups

def sync_lora_lr(optimizer):
    """Giữ LR của LoRA đúng bằng LORA_LR_SCALE × LR decoder sau mỗi bước của lịch cosine."""
    if len(optimizer.param_groups) > 1:
        optimizer.param_groups[1]['lr'] = optimizer.param_groups[0]['lr'] * LORA_LR_SCALE

def lora_state_dict(model):
    return {k: v.detach().cpu() for k, v in model.clip.vision_model.state_dict().items() if 'lora_' in k}

def load_lora_state(model, state):
    missing, unexpected = model.clip.vision_model.load_state_dict(state, strict=False)
    assert not unexpected and (not [k for k in missing if 'lora_' in k]), (missing, unexpected)

@contextmanager
def lora_disabled(model):
    """Embedding điều kiện (text, ảnh support) luôn tính bằng CLIP gốc, kể cả ở E4c."""
    modules = lora_modules(model)
    for m in modules:
        m.enabled = False
    try:
        yield
    finally:
        for m in modules:
            m.enabled = True

def vision_activations(model, pixel_values):
    """Output các layer extract_layers của CLIP vision, chỉ chạy tới layer cuối cần dùng."""
    vision = model.clip.vision_model
    hidden = vision.pre_layrnorm(vision.embeddings(pixel_values, interpolate_pos_encoding=True))
    activations = []
    for index, layer in enumerate(lora_layers(model)):
        if LORA_GRAD_CHECKPOINT and torch.is_grad_enabled():
            hidden = torch.utils.checkpoint.checkpoint(layer, hidden, None, None, use_reentrant=False)[0]
        else:
            hidden = layer(hidden, None, None)[0]
        if index in model.extract_layers:
            activations.append(hidden)
    return activations

def mix_conditionals(text_cond, visual_cond, text_weights):
    """cond = w·text + (1−w)·ảnh (giống cell PhraseCut+; định nghĩa sớm để segment dùng được khi chỉ đánh giá)."""
    weights = text_weights.to(text_cond.device, text_cond.dtype)
    return text_cond * weights + visual_cond * (1 - weights)

def segment(model, pixel_values, input_ids, attention_mask, support_pixel_values=None, text_weights=None):
    """Logits [B,352,352] cho train lẫn inference, model gốc (B1) hoặc CLIP vision có LoRA (E4c)."""
    size = pixel_values.shape[0]
    lora = has_lora(model)
    with torch.no_grad():
        if not lora:
            hidden = model.clip.vision_model(pixel_values=pixel_values, output_hidden_states=True, interpolate_pos_encoding=True).hidden_states
            activations = [hidden[i + 1] for i in model.extract_layers]
        with lora_disabled(model):
            cond = model.get_conditional_embeddings(batch_size=size, input_ids=input_ids, attention_mask=attention_mask)
            if support_pixel_values is not None:
                visual = model.get_conditional_embeddings(batch_size=size, conditional_pixel_values=support_pixel_values)
                cond = mix_conditionals(cond, visual, text_weights)
    if lora:
        activations = vision_activations(model, pixel_values)
    logits = model.decoder(activations, cond).logits
    return logits.reshape(size, *logits.shape[-2:])

def experiment_logits(model, pixel_values, input_ids=None, attention_mask=None, conditional_pixel_values=None):
    """E2/E3 forward: text hoặc ảnh support; với LoRA, embedding điều kiện vẫn dùng CLIP gốc."""
    if not has_lora(model):
        return model(pixel_values=pixel_values, input_ids=input_ids, attention_mask=attention_mask, conditional_pixel_values=conditional_pixel_values).logits
    if (input_ids is None) == (conditional_pixel_values is None):
        raise ValueError('Supply text or support images, not both')
    if input_ids is not None:
        return segment(model, pixel_values, input_ids, attention_mask)
    with torch.no_grad():
        activations = vision_activations(model, pixel_values)
        with lora_disabled(model):
            cond = model.get_conditional_embeddings(batch_size=pixel_values.shape[0], conditional_pixel_values=conditional_pixel_values)
    return model.decoder(activations, cond).logits


decoder_info = {"source": "pretrained", "checkpoint": None}
if DECODER_CHECKPOINT:
    decoder_path = Path(DECODER_CHECKPOINT)
    if not decoder_path.is_file():
        raise FileNotFoundError(f"DECODER_CHECKPOINT không tồn tại: {decoder_path}")
    checkpoint = torch.load(decoder_path, map_location="cpu", weights_only=True)
    if checkpoint.get("base_model_id") != MODEL_ID or checkpoint.get("base_model_revision") != MODEL_REVISION:
        raise ValueError("Checkpoint được train trên base model/revision khác MODEL_ID/MODEL_REVISION.")
    training = checkpoint.get("training") or {}
    if checkpoint.get("lora"):
        adapter = training.get("lora") or {}
        attach_lora(model, adapter.get("rank", 8), adapter.get("alpha", 16))
        load_lora_state(model, checkpoint["lora"])
    model.decoder.load_state_dict(checkpoint["decoder"])
    model.eval()
    decoder_info = {"source": "e4_trained_decoder" if training.get("arm") else "e1_trained_decoder", "checkpoint": str(decoder_path),
        "checkpoint_sha256": sha256(decoder_path), "epoch": checkpoint["epoch"],
        "global_step": checkpoint["global_step"], "epochs_planned": training.get("epochs"),
        "batch_size": training.get("batch_size"), "initialization": training.get("initialization"),
        "selected_train_queries": training.get("selected_train_queries"),
        # Checkpoint của bản E1 cũ không ghi train_mode: bản đó chỉ train bằng text.
        "train_mode": training.get("train_mode", "text_only" if training else None),
        "history": checkpoint.get("history"), "experiment_arm": training.get("arm"),
        "decoder_arch": training.get("decoder_arch", "clipseg"), "lora": training.get("lora")}
    if training.get("epochs") is not None and checkpoint["epoch"] != training["epochs"]:
        print(f"Cảnh báo: checkpoint epoch {checkpoint['epoch']}/{training['epochs']}, không phải epoch cuối.")
    del checkpoint
print("Decoder:", json.dumps({k: v for k, v in decoder_info.items() if k != "history"}, ensure_ascii=False))

def binary_metrics(probability, target, threshold):
    if probability.shape != target.shape or not target.any():
        raise ValueError("Shape không khớp hoặc GT rỗng.")
    prediction = probability >= threshold
    intersection = int(np.logical_and(prediction, target).sum())
    union = int(np.logical_or(prediction, target).sum())
    return {"intersection": intersection, "union": union, "iou": intersection / union,
            "dice": 2 * intersection / int(prediction.sum() + target.sum())}

def summarize(rows):
    if not rows:
        raise ValueError("Không có query để tính metric.")
    result = {"n_queries": len(rows), "n_images": len({r["image_id"] for r in rows}),
        "mean_sample_iou": float(np.mean([r["iou"] for r in rows])),
        "cumulative_iou": sum(r["intersection"] for r in rows) / sum(r["union"] for r in rows),
        "mean_dice": float(np.mean([r["dice"] for r in rows]))}
    for cutoff in [0.3, 0.5, 0.7, 0.9]:
        result[f"precision_at_{cutoff}"] = float(np.mean([r["iou"] >= cutoff for r in rows]))
    if "pixel_ap" in rows[0]:
        result["mean_sample_pixel_ap"] = float(np.mean([r["pixel_ap"] for r in rows]))
    return result

@torch.inference_mode()
def predict(image, prompts=None, supports=None):
    """Xác suất [N,H,W] ở kích thước gốc và [N,352,352] để chấm theo paper, cho N prompt trên một ảnh query."""
    if (prompts is None) == (supports is None):
        raise ValueError("Truyền text prompts hoặc support images, không cả hai.")
    count = len(prompts) if prompts is not None else len(supports)
    # Tiền xử lý như CLIPSeg gốc (paper_pixel_values), cho cả ảnh query lẫn ảnh support.
    inputs = {"pixel_values": paper_pixel_values(image).unsqueeze(0).repeat(count, 1, 1, 1)}
    if prompts is not None:
        inputs.update(processor.tokenizer(text=list(prompts), padding=True, truncation=True, return_tensors="pt"))
    else:
        inputs["conditional_pixel_values"] = torch.stack([s if isinstance(s, torch.Tensor) else paper_pixel_values(s)
                                                         for s in supports])
    inputs = {k: v.to(device) for k, v in inputs.items()}
    synchronize()
    started = time.perf_counter()
    context = torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP else nullcontext()
    with context:
        logits = experiment_logits(model, **inputs)
    logits = logits.float().reshape(count, *logits.shape[-2:])
    probabilities = F.interpolate(logits.unsqueeze(1), size=(image.height, image.width),
        mode="bilinear", align_corners=False).squeeze(1).sigmoid()
    synchronize()
    seconds = time.perf_counter() - started
    values = probabilities.cpu().numpy()
    if not np.isfinite(values).all():
        raise ValueError("Prediction chứa NaN/Inf; thử USE_AMP=False.")
    return values, logits.sigmoid().cpu().numpy(), seconds

paper_stats = defaultdict(list)   # biến thể -> tp/fp/fn/tn tại 49 ngưỡng của từng dòng metric

def evaluate(probability, target, base, probability_352):
    """Metric tại THRESHOLD (kèm pixel AP), các dòng quét ngưỡng, và stats theo protocol paper ở 352×352."""
    row = {**base, "threshold": THRESHOLD, **binary_metrics(probability, target, THRESHOLD),
           "pixel_ap": float(average_precision_score(target.ravel(), probability.ravel()))}
    sweep = [{**base, "threshold": t, **binary_metrics(probability, target, t)} for t in THRESHOLD_GRID]
    paper_stats[base["variant"]].append(paper_threshold_stats(probability_352, paper_target(target)))
    return row, sweep

def save_paper_stats():
    np.savez_compressed(OUT / "paper_threshold_stats.npz", thresholds=PAPER_THRESHOLDS,
        **{f"variant_{i}": np.stack(paper_stats[v]) for i, v in enumerate(paper_stats)})
    return {f"variant_{i}": v for i, v in enumerate(paper_stats)}

def variant_report(rows, sweep_rows, stats):
    """Ngưỡng cố định trên mọi query; ngưỡng bổ sung chọn chỉ trên calibration."""
    table = []
    for threshold in THRESHOLD_GRID:
        subset = [r for r in sweep_rows if r["group"] == "calibration" and r["threshold"] == threshold]
        table.append({"threshold": threshold, **summarize(subset)})
    best = max(table, key=lambda r: (r["mean_sample_iou"], -r["threshold"]))["threshold"]
    return {
        "all_fixed_threshold": summarize(rows),
        "calibration_fixed_threshold": summarize([r for r in rows if r["group"] == "calibration"]),
        "evaluation_fixed_threshold": summarize([r for r in rows if r["group"] == "evaluation"]),
        "calibration_selected_threshold": best,
        "evaluation_calibrated_threshold": summarize([r for r in sweep_rows
            if r["group"] == "evaluation" and r["threshold"] == best]),
        "paper_protocol": paper_metrics(stats),
    }, table

def paired_comparison(base_iou, other_iou, image_of, n_bootstrap, tie_tolerance, seed):
    """ΔIoU theo query (other − base), CI 95% bootstrap theo ảnh, số query thắng/hòa/thua."""
    query_ids = sorted(base_iou)
    if set(query_ids) != set(other_iou):
        raise ValueError("Hai biến thể phải có cùng tập query.")
    deltas = np.array([other_iou[q] - base_iou[q] for q in query_ids])
    groups = defaultdict(list)
    for index, query_id in enumerate(query_ids):
        groups[image_of[query_id]].append(index)
    clusters = [np.array(indices) for _, indices in sorted(groups.items())]
    rng = np.random.default_rng(seed)
    boot = [deltas[np.concatenate([clusters[i] for i in rng.integers(0, len(clusters), len(clusters))])].mean()
            for _ in range(n_bootstrap)]
    return {"n_queries": len(deltas), "delta_mean_iou": float(deltas.mean()),
            "ci95_low": float(np.percentile(boot, 2.5)), "ci95_high": float(np.percentile(boot, 97.5)),
            "wins": int((deltas > tie_tolerance).sum()), "ties": int((np.abs(deltas) <= tie_tolerance).sum()),
            "losses": int((deltas < -tie_tolerance).sum())}

def overlay(image, probability):
    rgb = np.asarray(image).astype(np.float32).copy()
    mask = probability >= THRESHOLD
    rgb[mask] = 0.55 * rgb[mask] + 0.45 * np.array([255, 60, 60])
    return rgb.astype(np.uint8)

def load_rgb(image_id):
    with Image.open(image_paths[image_id]) as raw:
        return raw.convert("RGB")

def save_mask(probability, variant, stem):
    if SAVE_MASKS:
        folder = OUT / "masks" / variant
        folder.mkdir(parents=True, exist_ok=True)
        Image.fromarray((probability >= THRESHOLD).astype(np.uint8) * 255).save(folder / f"{stem}.png")

def table_row(name, report, comparison):
    fixed = report["all_fixed_threshold"]
    paper = report["paper_protocol"]
    return {"variant": name, "rows": fixed["n_queries"],
        "paper_miou_0.3": paper["miou_0.3"], "paper_fgiou_0.3": paper["fgiou_0.3"], "paper_ap": paper["ap"],
        "mean_iou": fixed["mean_sample_iou"],
        "cumulative_iou": fixed["cumulative_iou"], "mean_dice": fixed["mean_dice"],
        "mean_pixel_ap": fixed["mean_sample_pixel_ap"], "precision_at_0.5": fixed["precision_at_0.5"],
        "calibrated_threshold": report["calibration_selected_threshold"],
        "evaluation_iou_fixed": report["evaluation_fixed_threshold"]["mean_sample_iou"],
        "evaluation_iou_calibrated": report["evaluation_calibrated_threshold"]["mean_sample_iou"],
        **{key: (comparison or {}).get(key) for key in
           ["delta_mean_iou", "ci95_low", "ci95_high", "wins", "ties", "losses"]}}

def print_table(rows):
    columns = ["variant", "rows", "paper_miou_0.3", "paper_fgiou_0.3", "paper_ap", "mean_iou",
               "delta_mean_iou", "ci95_low", "ci95_high", "wins", "losses"]
    print(" | ".join(columns))
    for row in rows:
        print(" | ".join(f"{row[c]:.4f}" if isinstance(row[c], float) else str(row[c]) for c in columns))

def run_metadata(extra):
    inference_seconds = sum(b["forward_resize_seconds"] for b in batch_times)
    prompts = sum(b["prompts"] for b in batch_times)
    return {**extra, "decoder": decoder_info,
        "model_id": MODEL_ID, "requested_revision": MODEL_REVISION if not MODEL_PATH else None,
        "model_source": model_source, "resolved_commit": getattr(model.config, "_commit_hash", None),
        "phrasecut_api_commit": PHRASECUT_API_COMMIT,
        "annotation_sha256": sha256(annotation_path), "image_metadata_sha256": sha256(info_path),
        "seed": SEED, "threshold": THRESHOLD, "threshold_grid": THRESHOLD_GRID,
        "calibration_fraction": CALIBRATION_FRACTION, "n_bootstrap": N_BOOTSTRAP, "tie_tolerance": TIE_TOLERANCE,
        "precision": "fp16_autocast" if USE_AMP else "fp32", "device": str(device),
        "gpu_name": torch.cuda.get_device_name(0) if device.type == "cuda" else None,
        "torch_cuda_version": torch.version.cuda, "python": platform.python_version(), "versions": versions,
        "image_processor": processor.image_processor.to_dict(),
        "forward_resize_seconds_total": inference_seconds,
        "forward_resize_ms_per_prompt": 1000 * inference_seconds / prompts,
        "wall_seconds_inference_metrics_saving": elapsed_all,
        "peak_allocated_gpu_mb": torch.cuda.max_memory_allocated(device) / 1024**2 if device.type == "cuda" else None,
        "timing_note": "Two warmup batches; forward+resize+sigmoid timed with CUDA sync; excludes preprocessing and metric/save.",
    }


## 6. Chạy 4 template trên mọi query

Mỗi ảnh gom (query × template) thành batch. Mask lưu tại `masks/<t0..t3>/<task_id>.png`
(bảng ánh xạ key → template ở `run.json`). Hình minh họa đặt 4 template cạnh nhau cho cùng query.


In [ ]:
assert BATCH_SIZE >= 1
assert all("{phrase}" in t for t in TEMPLATES) and len(set(TEMPLATES)) == len(TEMPLATES)
assert BASELINE_TEMPLATE in TEMPLATES
template_keys = {template: f"t{i}" for i, template in enumerate(TEMPLATES)}
(OUT / "visualizations").mkdir(exist_ok=True)

tasks_by_image = defaultdict(list)
for task in selected:
    tasks_by_image[int(task["image_id"])].append(task)
warmup_image = load_rgb(image_ids[0])
warmup_prompts = [TEMPLATES[0].format(phrase=t["phrase"]) for t in tasks_by_image[image_ids[0]]][:BATCH_SIZE]
for _ in range(2):
    predict(warmup_image, prompts=warmup_prompts)
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats(device)

def save_template_figure(image, target, task, probabilities, ious, index):
    fig, axes = plt.subplots(1, 2 + len(TEMPLATES), figsize=(4 * (2 + len(TEMPLATES)), 4))
    axes[0].imshow(image); axes[0].set_title("Ảnh")
    axes[1].imshow(target, cmap="gray", vmin=0, vmax=1); axes[1].set_title("Ground truth")
    for ax, template in zip(axes[2:], TEMPLATES):
        ax.imshow(overlay(image, probabilities[template]))
        ax.set_title(f"{template_keys[template]}: IoU={ious[template]:.3f}\n{template.format(phrase='…')}")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"{task['task_id']} — {task['phrase']}")
    fig.tight_layout()
    fig.savefig(OUT / "visualizations" / f"{index:02d}_{task['task_id']}.png", dpi=100, bbox_inches="tight")
    plt.close(fig)

rows, sweep_rows, batch_times = [], [], []
visual_count = 0
started_all = time.perf_counter()
for image_id in tqdm(image_ids, desc="E3 images"):
    image = load_rgb(image_id)
    current = tasks_by_image[image_id]
    targets = {str(t["task_id"]): rasterize_gt(t, image.width, image.height) for t in current}
    jobs = [(task, template) for task in current for template in TEMPLATES]
    kept = defaultdict(dict)      # xác suất của các query được vẽ hình
    ious = defaultdict(dict)
    for start in range(0, len(jobs), BATCH_SIZE):
        chunk = jobs[start:start + BATCH_SIZE]
        probabilities, probabilities_352, seconds = predict(image, prompts=[tpl.format(phrase=t["phrase"]) for t, tpl in chunk])
        batch_times.append({"prompts": len(chunk), "forward_resize_seconds": seconds})
        for (task, template), probability, probability_352 in zip(chunk, probabilities, probabilities_352):
            task_id = str(task["task_id"])
            base = {"task_id": task_id, "image_id": image_id, "phrase": task["phrase"],
                "variant": template, "variant_key": template_keys[template],
                "prompt": template.format(phrase=task["phrase"]), "group": group_for_image[image_id],
                "width": image.width, "height": image.height}
            row, sweep = evaluate(probability, targets[task_id], base, probability_352)
            rows.append(row)
            sweep_rows.extend(sweep)
            save_mask(probability, template_keys[template], task_id)
            ious[task_id][template] = row["iou"]
            kept[task_id][template] = probability
    for task in current:
        task_id = str(task["task_id"])
        if visual_count < N_VISUALIZATIONS:
            save_template_figure(image, targets[task_id], task, kept[task_id], ious[task_id], visual_count)
            visual_count += 1
    write_csv(OUT / "sample_metrics.csv", rows)
elapsed_all = time.perf_counter() - started_all
assert len(rows) == len(selected) * len(TEMPLATES), "Chưa đánh giá đủ query × template."
print(f"Completed {len(selected)} queries × {len(TEMPLATES)} templates; wall time {elapsed_all:.1f}s")


## 7. Bảng E3

`e3_template_table.csv` là bảng chính: mỗi template một dòng, metric tại t=0,3 trên toàn bộ query đã chọn,
ΔIoU so với `{phrase}` kèm CI 95% và số query thắng/hòa/thua. CI chứa 0 nghĩa là chưa đủ bằng chứng
template đó khác `{phrase}`. Cột calibrated chỉ để tham khảo, ngưỡng được chọn trên calibration.


In [ ]:
image_of = {str(t["task_id"]): int(t["image_id"]) for t in selected}
reports, calibration_rows, table = {}, [], []
iou_by_template = {}
for template in TEMPLATES:
    report, calibration_table = variant_report([r for r in rows if r["variant"] == template],
                                               [r for r in sweep_rows if r["variant"] == template],
                                               paper_stats[template])
    reports[template] = report
    calibration_rows += [{"variant": template, **r} for r in calibration_table]
    iou_by_template[template] = {r["task_id"]: r["iou"] for r in rows if r["variant"] == template}
for template in TEMPLATES:
    comparison = None if template == BASELINE_TEMPLATE else paired_comparison(
        iou_by_template[BASELINE_TEMPLATE], iou_by_template[template], image_of, N_BOOTSTRAP, TIE_TOLERANCE, SEED)
    reports[template]["paired_vs_baseline"] = comparison
    table.append({"variant_key": template_keys[template], **table_row(template, reports[template], comparison)})
write_csv(OUT / "calibration_thresholds.csv", calibration_rows)
write_csv(OUT / "e3_template_table.csv", table)
paper_stats_keys = save_paper_stats()
summary = {"experiment": "E3_prompt_templates", "decoder": decoder_info, "dataset": "VGPhraseCut", "split": EVAL_SPLIT,
    "n_queries": len(selected), "complete_official_split": len(selected) == len(tasks),
    "baseline_template": BASELINE_TEMPLATE, "templates": reports,
    "paper_threshold_stats_keys": paper_stats_keys,
    "metric_note": ("paper_protocol per template follows CLIPSeg score.py/FixedIntervalMetrics at 352x352; "
                    "other metrics at original resolution, IoU averaged per query at t=0.3; ΔIoU paired per query; "
                    "CI is image-level bootstrap.")}
write_json(OUT / "summary.json", summary)
write_json(OUT / "run.json", run_metadata({"experiment": "E3_prompt_templates", "max_queries": MAX_QUERIES,
    "batch_size": BATCH_SIZE, "templates": template_keys, "baseline_template": BASELINE_TEMPLATE,
    "selected_tasks_sha256": sha256(OUT / "selected_tasks.json"),
    "calibration_image_ids": sorted(calibration_ids),
    "evaluation_image_ids": sorted(set(image_ids) - calibration_ids)}))
print_table(table)


## 8. Xem hình và đóng gói

Tải `clipseg_e3_prompt_templates.zip` trong Kaggle Output.


In [ ]:
for path in sorted((OUT / "visualizations").glob("*.png"))[:N_VISUALIZATIONS]:
    with Image.open(path) as image:
        display(image.copy())
archive = WORK_ROOT / f"{OUTPUT_NAME}.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as package:
    for path in sorted(OUT.rglob("*")):
        if path.is_file():
            package.write(path, arcname=path.relative_to(OUT).as_posix())
print("Result folder:", OUT)
print("Download:", archive)


## Nguồn và giới hạn

- [CLIPSeg / CVPR 2022](https://github.com/timojl/clipseg), [checkpoint HF](https://huggingface.co/CIDAS/clipseg-rd64-refined).
- [PhraseCutDataset](https://github.com/ChenyunWu/PhraseCutDataset): annotation, URL ảnh, split và quy ước polygon/mask.
- [CLIPSeg trong Transformers 4.55.4](https://huggingface.co/docs/transformers/v4.55.4/en/model_doc/clipseg).

Checkpoint pretrained, không train. PhraseCut liên quan tới dữ liệu huấn luyện CLIPSeg nên không gọi là zero-shot.
Google Drive/Visual Genome có thể gián đoạn; Add Input dataset chứa hai file JSON (và ảnh) là đường dự phòng.
